In [1]:
# Reads in FITS files from the MCT and checks them
# frames are taken at temperatures 55K and 90K

In [1]:
from astropy.io import fits
import numpy as np
import glob
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
def save_individ_fits_frame_inspection_plot(file_name, index, out_path=None):
    """Load FITS primary HDU, plot full frame and center row/column profiles, save PNG."""
    if out_path is None:
        out_path = f"junk_{index:02d}.png"
    with fits.open(file_name) as hdul:
        data = hdul[0].data
    print(data.shape)
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    im0 = axs[0].imshow(data, origin="lower")
    axs[0].set_title("Full Image")
    fig.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

    middle_row = data[data.shape[0] // 2, :]
    axs[1].plot(middle_row)
    axs[1].set_title("Middle Row Profile")
    axs[1].set_xlabel("Column")
    axs[1].set_ylabel("Value")

    central_col = data[:, data.shape[0] // 2]
    axs[2].plot(central_col)
    axs[2].set_title("Central col Profile")
    axs[2].set_xlabel("Row")
    axs[2].set_ylabel("Value")

    fig.suptitle(f"{file_name}\nmean: {np.mean(data):.2f}, std: {np.std(data):.2f}")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    plt.savefig(out_path)
    plt.close(fig)
    print("Wrote ", out_path)

    return


In [ ]:
def write_hot_pixel_map_pngs_from_fits_list(
    file_name_list,
    width_col=64,
    n_cols=32,
    out_prefix="frame_hot_",
    threshold_sigma=5,
    dpi=150,
):
    """
    Median-subtract each vertical readout band, threshold frame, save binary hot-pixel map PNGs.
    """
    for file_name_num, file_name in enumerate(file_name_list):
        print(file_name)
        with fits.open(file_name) as hdul:
            frame_data = np.asarray(hdul[0].data, dtype=float).copy()

        for col_num in range(n_cols):
            idx1, idx2 = width_col * col_num, width_col * (col_num + 1)
            col_data = frame_data[:, idx1:idx2]
            median_val = np.median(col_data)
            frame_data[:, idx1:idx2] = col_data - median_val

        threshold = np.median(frame_data) + threshold_sigma * np.std(frame_data)
        pix_map = np.zeros_like(frame_data, dtype=np.uint8)
        pix_map[frame_data > threshold] = 0
        pix_map[frame_data < threshold] = 1

        file_name_plot = f"{out_prefix}{file_name_num}.png"
        plt.figure(figsize=(5, 5))
        plt.imshow(pix_map, origin="lower", cmap="gray")
        plt.title(f"Hot Pixel Map: {file_name_num}")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(file_name_plot, dpi=dpi)
        plt.close()
        n_hot = int(np.sum(frame_data > threshold))
        print(f"Wrote {file_name_plot} ({n_hot} hot pixels)")

    return


In [ ]:
def per_pixel_linear_fit_slopes_intercepts(
    file_name_list,
    length_edge=2048,
    first_idx_to_fit=20,
    readout_width=64,
):
    '''
    Stack FITS frames, median-correct by readout column band, fit linear drift vs frame index.

    INPUTS: 
    file_name_list : sequence of str
        Paths to FITS files
    length_edge : int
        Size of the frame in pixels (one edge)
    first_idx_to_fit : int
        Drop frame indices [0, first_idx_to_fit) before the per-pixel lstsq.
    readout_width : int
        Width in columns of each readout block for per-frame median subtraction (if applied)

    OUTPUTS:
    slopes, intercepts : ndarray, shape (length_edge, length_edge)
        Per-pixel linear fit coefficients vs frame index (same x for every pixel).
    '''

    paths = list(file_name_list)
    paths = sorted(paths)
    if not paths:
        raise ValueError("file_name_list is empty")

    # check size of frame
    with fits.open(paths[0]) as hdul:
        data_test = hdul[0].data
    ny, nx = int(data_test.shape[0]), int(data_test.shape[1])
    if length_edge > ny or length_edge > nx:
        raise ValueError(f"length_edge {length_edge} exceeds frame shape {(ny, nx)}")

    # put all the frames into a cube
    data_cube = np.zeros((len(paths), ny, nx), dtype=np.asarray(data_test).dtype)
    for i, file_name in enumerate(paths):
        with fits.open(file_name) as hdul:
            data_cube[i, :, :] = hdul[0].data

    x = np.arange(len(paths), dtype=float)

    data_cube_corr = data_cube[:, :length_edge, :length_edge].copy()

    '''
    for col_start in range(0, length_edge, readout_width):
        col_end = min(col_start + readout_width, length_edge)
        readout = data_cube_corr[:, :, col_start:col_end]
        readout_median = np.median(readout, axis=(1, 2), keepdims=True)
        data_cube_corr[:, :, col_start:col_end] = readout - readout_median
    '''

    data_cube_corr_trunc = data_cube_corr[first_idx_to_fit:, :, :]
    x_trunc = x[first_idx_to_fit:]
    y = data_cube_corr_trunc.reshape(len(x_trunc), -1)
    a = np.vstack([x_trunc, np.ones_like(x_trunc)]).T
    coeffs, _, _, _ = np.linalg.lstsq(a, y, rcond=None)
    slopes = coeffs[0].reshape(length_edge, length_edge)
    intercepts = coeffs[1].reshape(length_edge, length_edge)
    
    return slopes, intercepts

In [2]:
stem_55k = '/Users/eckhartspalding/Documents/git.repos/nice2/data/20260410152426/'
stem_90k = '/Users/eckhartspalding/Documents/git.repos/nice2/data/20260410152426/'

In [3]:
file_name_list_55k = glob.glob(stem_55k + '*.fits')
file_name_list_55k = sorted(file_name_list_55k)

file_name_list_90k = glob.glob(stem_90k + '*.fits')
file_name_list_90k = sorted(file_name_list_90k)

In [ ]:
# initial inspection of each frame

for file_name_num, file_name in enumerate(file_name_list_55k):
    save_individ_fits_frame_inspection_plot(file_name, file_name_num)

(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)


/var/folders/wb/zn41c4yx58z1ktmcwgv62zyr0000gn/T/ipykernel_50209/2875213860.py:7: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(1, 3, figsize=(18, 5))


(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)
(2048, 2048)


<Figure size 1800x500 with 0 Axes>

<Figure size 1800x500 with 0 Axes>

...

<Figure size 1800x500 with 0 Axes>

<Figure size 1800x500 with 0 Axes>

<Figure size 1800x500 with 0 Axes>

<Figure size 1800x500 with 0 Axes>

<Figure size 1800x500 with 0 Axes>

In [ ]:
# write out hot pixel maps
write_hot_pixel_map_pngs_from_fits_list(file_name_list)


(2048, 2048)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317161526_R01_M01_N01.fits
Wrote frame_hot_0.png (4194304 hot pixels)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317165058_R01_M01_N01.fits
Wrote frame_hot_1.png (4187421 hot pixels)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317165250_R01_M01_N01.fits
Wrote frame_hot_2.png (4186198 hot pixels)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317161516_R01_M01_N01.fits
Wrote frame_hot_3.png (4194304 hot pixels)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317165304_R01_M01_N01.fits
Wrote frame_hot_4.png (4186419 hot pixels)
/Users/eckhartspalding/Documents/git.repos/nice/data/20270317_mct_test_readouts/H2RG_20260317161153_R01_M01_N01.fits
Wrote frame_hot_5.png (4194304 hot pixels)
/Users/eckhartspalding/Docu